# LLM council forecasting: reproduce the results

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sweksha-cloud/llm-council-forecasting/blob/main/notebooks/reproduce.ipynb)

**Research question:** does a deliberating council of small LLMs forecast better than a simple average of its members' probabilities, and what explains the difference?

This notebook clones the repository, runs the test suite, rebuilds every result from the committed data (`make reproduce` fails if anything differs from what's committed), and shows the main tables. No GPU or model is needed: the forecasts were made on SJSU's GPU server and are stored in `data/forecasts.csv`.

Project: Sweksha Shaw, San José State University (CS 180), advised by Prof. William Andreopoulos. Results page: https://sweksha-cloud.github.io/llm-council-forecasting/

In [ ]:
!git clone -q https://github.com/sweksha-cloud/llm-council-forecasting.git
%cd llm-council-forecasting
!pip install -q -r requirements.txt

## 1. Tests
Scoring checked against hand-computed values, the dashboard against the Python scores, and the data against its integrity rules.

In [ ]:
!python -m pytest -q

## 2. Rebuild every result and check it matches
`make reproduce` re-imports the earlier council results from their pinned commit, rebuilds `results/` and the dashboard, and fails if any committed result changes.

In [ ]:
!make reproduce 2>&1 | tail -3

## 3. Main comparisons
Paired Brier difference, council minus reference, with a 95% bootstrap interval stratified by source (negative = council better). The pre-registered sample is `settled-all` (1,039 questions); it fills in once the settled forecasts are merged.

In [ ]:
import pandas as pd
from pathlib import Path

def show(path, cols=None):
    p = Path(path)
    if not p.exists() or p.stat().st_size == 0:
        print(f"{path}: not available yet")
        return None
    df = pd.read_csv(p)
    return df[cols] if cols else df

show("results/settled-all/comparisons.csv")

In [ ]:
cols = ["setup", "answered", "brier", "log_score", "accuracy", "ece", "auroc", "skill_vs_market_paired", "skill_ci_low", "skill_ci_high"]
show("results/settled-all/scores.csv", [c for c in cols if c in pd.read_csv("results/settled-all/scores.csv").columns])

## 4. How each chair combines its members
Weights that best reproduce the chair's forecast as a weighted average of the members' forecasts in log-odds, and whether a chair favours its own model (live set, exploratory; the settled-set test is pre-registered).

In [ ]:
display(show("results/live/chair_weights.csv"))
show("results/live/self_preference.csv")

## 5. The earlier council results, re-scored
The earlier study's councils against the market price and an always-No baseline, after fixing its scoring bugs (see `docs/leakage.md`).

In [ ]:
show('results/aye-2026-new/comparisons.csv')

## 6. Use the benchmark for your own forecaster
`benchmark/questions_v1.csv` holds 1,185 resolved questions with the market price at the `as_of` date. Ask your model each question as of that date, then score it here. Below, the market itself is scored as the reference.

In [ ]:
from scoring.proper import mean_brier

bench = pd.read_csv("benchmark/questions_v1.csv")
settled = bench[bench.split == "settled"]
print(len(settled), "settled questions;", f"{settled.outcome.mean():.1%} resolved Yes")
print("Market Brier:", round(mean_brier(settled.market_price.tolist(), settled.outcome.tolist()), 4))
print("Always-No Brier:", round(mean_brier([0.0] * len(settled), settled.outcome.tolist()), 4))

# Replace this with your model's probabilities, one per question_id:
my_forecasts = dict(zip(settled.question_id, [0.5] * len(settled)))
ps = [my_forecasts[q] for q in settled.question_id]
print("My Brier:", round(mean_brier(ps, settled.outcome.tolist()), 4))

In [ ]:
import matplotlib.pyplot as plt
bins = pd.cut(settled.market_price, [i / 10 for i in range(11)], include_lowest=True)
rel = settled.groupby(bins, observed=True).agg(forecast=("market_price", "mean"), observed=("outcome", "mean"), n=("outcome", "size"))
plt.plot([0, 1], [0, 1], "--", color="grey")
plt.plot(rel.forecast, rel.observed, "o-")
plt.xlabel("Market price 7 days before close"); plt.ylabel("Share resolved Yes"); plt.title("Market calibration, settled questions")
plt.show()